In [1]:
import sys
sys.path.append("..")
sys.path.append("../scripts")

In [2]:
from ms2maccs import MS2Data, MACCSModel, collate_fn
from utils import calc_tanimoto
from matchms.importing import load_from_mgf

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import ConcatDataset

In [3]:
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    total_pred = []
    total_true = []

    with torch.no_grad():
        for submaccs, maccs in loader: 
            maccs = maccs.to(DEVICE)
            logits = model(submaccs)

            loss = loss_fn(logits, maccs)
            total_loss += loss.item() * len(maccs)
            pred_maccs = (torch.sigmoid(logits) >= 0.5).float()
            total_pred.append(pred_maccs.cpu())
            total_true.append(maccs.cpu())

        avg_loss = total_loss / len(loader.dataset)
        tanimoto = calc_tanimoto(torch.cat(total_pred), torch.cat(total_true))
        
        return avg_loss, tanimoto

In [16]:
def train(model, train_loader, val_loader):
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)

    best_tanimoto = -1.0
    best_state = None # maybe tanimoto is not the best indication alone

    print(f"\nTraining on {DEVICE}  |  epochs={EPOCHS}  |  batch={BATCH_SIZE}")
    print("." * 48)
    print(f"{'Epoch':>6}  {'Train Loss':>11}  {'Val Loss':>9}  {'Val Tanimoto':>13}")
    print("-" * 48)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss = 0.0

        for submaccs, maccs in train_loader:
            maccs = maccs.to(DEVICE)
            optimizer.zero_grad()
            logits = model(submaccs)
            loss = loss_fn(logits, maccs)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item() * len(maccs)

        train_loss = epoch_loss / len(train_loader.dataset)
        val_loss, val_tanimoto = evaluate(model, val_loader, loss_fn)
        scheduler.step(val_tanimoto)

        print(f"{epoch:>6}  {train_loss:>11.4f}  {val_loss:>9.4f}  {val_tanimoto:>13.4f}")

        if val_tanimoto > best_tanimoto:
            best_tanimoto = val_tanimoto
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

    print(f"\nBest validation Tanimoto: {best_tanimoto:.4f}")
    model.load_state_dict(best_state)
    return model

In [11]:
# Dataset
MGF_TRAIN = "../ms2_data/train_specs_H_p_n_mode.mgf"
MGF_VAL_P = "../ms2_data/val_specs_H_p_mode.mgf"
MGF_VAL_N = "../ms2_data/val_specs_H_n_mode.mgf"
FP_BIT_MAP_P_MODE_PATH = "../fp_bit_maps/fp_bit_map_H_p_mode.pkl"
FP_BIT_MAP_N_MODE_PATH = "../fp_bit_maps/fp_bit_map_H_n_mode.pkl"
BATCH_SIZE = 64
EPOCHS = 20
DEVICE = "cuda"

# Model
d_model = 512
nhead = 8
num_layers = 5
dropout = 0.15

# Optimizer 
lr = 0.00013182100938388214
weight_decay = 0.003093223263612351

In [24]:
ms2train = MS2Data(MGF_TRAIN, FP_BIT_MAP_P_MODE_PATH, FP_BIT_MAP_N_MODE_PATH)
ms2train_loader = DataLoader(ms2train, batch_size=BATCH_SIZE, collate_fn=collate_fn, shuffle=True)

Processing train_specs_H_p_n_mode.mgf: 27170it [01:41, 267.69it/s]


In [25]:
ms2val_p = MS2Data(MGF_VAL_P, FP_BIT_MAP_P_MODE_PATH, FP_BIT_MAP_N_MODE_PATH)
ms2val_n = MS2Data(MGF_VAL_N, FP_BIT_MAP_P_MODE_PATH, FP_BIT_MAP_N_MODE_PATH)
ms2val = ConcatDataset([ms2val_p, ms2val_n])
ms2val_loader = DataLoader(ms2val_p, batch_size=BATCH_SIZE, collate_fn=collate_fn)

Processing val_specs_H_p_mode.mgf: 4409it [00:14, 305.41it/s]
Processing val_specs_H_n_mode.mgf: 990it [00:02, 399.32it/s]


In [26]:
model = MACCSModel(d_model=d_model, nhead=nhead, num_layers=num_layers, dropout=dropout).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters : {total_params:,}")

Trainable parameters : 16,197,799


In [27]:
model = train(model, ms2train_loader, ms2val_loader)


Training on cuda  |  epochs=20  |  batch=64
................................................
 Epoch   Train Loss   Val Loss   Val Tanimoto
------------------------------------------------
     1       0.3165     0.3094         0.5088
     2       0.2687     0.3002         0.5200
     3       0.2529     0.3030         0.5299
     4       0.2408     0.2959         0.5414
     5       0.2315     0.2896         0.5457
     6       0.2232     0.2878         0.5469
     7       0.2166     0.2810         0.5505
     8       0.2092     0.2855         0.5511
     9       0.2021     0.2875         0.5538
    10       0.1959     0.2897         0.5585
    11       0.1903     0.2931         0.5552
    12       0.1845     0.2921         0.5554
    13       0.1789     0.3036         0.5545
    14       0.1736     0.2964         0.5565
    15       0.1682     0.2994         0.5606
    16       0.1629     0.3045         0.5556
    17       0.1569     0.3088         0.5546
    18       0.1528     0.308

In [28]:
torch.save(model.state_dict(), "../models/standard_model.pt")